# RQ3: Yambda Encoder Comparison With Fixed dVAE

Compare teammate-provided E5 and BGE embeddings with fixed-length dVAE and the same lightweight seqrec setup used in `01_run_experiments.ipynb`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path('/home/jupyter/project/semantic_ids')
if not ROOT.exists():
    ROOT = Path.cwd().resolve()
os.chdir(ROOT)

PY = sys.executable
RAW_DIR = ROOT / 'data' / 'yambda_encoder_raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS = {
    'e5': RAW_DIR / 'embeddings_e5.parquet',
    'bge': RAW_DIR / 'embeddings_bge.parquet',
}
INTERACTIONS = RAW_DIR / 'interactions.parquet'

print('ROOT:', ROOT)
print('RAW_DIR:', RAW_DIR)

## Put New Embeddings In Place

Expected project paths:

```text
data/yambda_encoder_raw/embeddings_e5.parquet
data/yambda_encoder_raw/embeddings_bge.parquet
```

On the local Mac they were copied from:

```text
/Users/ad-scherbakov/Downloads/new/embeddings_e5.parquet
/Users/ad-scherbakov/Downloads/new/embeddings_bge.parquet
```

In [ ]:
# Optional local copy. On Jupyter, upload/copy files to RAW_DIR manually if these source paths do not exist.

local_sources = {
    'e5': Path('/Users/ad-scherbakov/Downloads/new/embeddings_e5.parquet'),
    'bge': Path('/Users/ad-scherbakov/Downloads/new/embeddings_bge.parquet'),
}

for name, dst in EMBEDDINGS.items():
    src = local_sources[name]
    if not dst.exists() and src.exists():
        shutil.copy2(src, dst)
    if not dst.exists():
        raise FileNotFoundError(f'Missing {name} embeddings: {dst}')
    print(name, dst, 'MB=', round(dst.stat().st_size / 1e6, 2))

## Reuse Or Download Yambda Interactions

Old interactions are enough. Old prepared dVAE item tables are not enough because they contain old embeddings.

In [ ]:
interaction_candidates = [
    INTERACTIONS,
    ROOT / 'data' / 'yambda' / 'interactions.parquet',
    ROOT / 'data' / 'RQ_album_artist_anchor' / 'yambda' / 'subset_interactions.parquet',
]

src_interactions = next((p for p in interaction_candidates if p.exists()), None)
if src_interactions is not None:
    if src_interactions.resolve() != INTERACTIONS.resolve():
        shutil.copy2(src_interactions, INTERACTIONS)
    print('Reused interactions:', src_interactions)
else:
    print('Downloading yandex/yambda likes.parquet...')
    from datasets import load_dataset

    ds = load_dataset(
        'yandex/yambda',
        data_dir='flat/5b',
        data_files='likes.parquet',
    )['train'].to_polars()
    ds.write_parquet(INTERACTIONS)

print('Interactions:', INTERACTIONS, 'MB=', round(INTERACTIONS.stat().st_size / 1e6, 2))

## Check Embeddings

This checks columns, row counts, item duplicates, embedding dimension, norms, finite values, and cosine quantiles on a sample.

In [ ]:
check_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.check_embeddings',
    '--embedding', f'e5={EMBEDDINGS["e5"]}',
    '--embedding', f'bge={EMBEDDINGS["bge"]}',
    '--output', 'results/RQ_encoder_comparison_yambda/embedding_checks.json',
]

print('+', ' '.join(check_cmd))
subprocess.run(check_cmd, check=True)

## Smoke Test

Small fixed-dVAE run for both encoders: 1000 users, 5000 core items, 1 epoch, no seqrec.

In [ ]:
smoke_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.run_encoder_experiment',
    '--interactions', str(INTERACTIONS),
    '--embedding', f'e5={EMBEDDINGS["e5"]}',
    '--embedding', f'bge={EMBEDDINGS["bge"]}',
    '--test-interval-days', '7',
    '--work-data-dir', 'data/RQ_encoder_comparison/yambda_fixed',
    '--results-dir', 'results/RQ_encoder_comparison_yambda_fixed',
    '--dvae-config', 'configs/RQ_encoder_comparison/yambda_fixed_dvae.yaml',
    '--seqrec-config', 'configs/RQ_encoder_comparison/seqrec_yambda_fixed.yaml',
    '--stages', 'prepare,dvae,sid_metrics',
    '--max-users', '1000',
    '--max-core-items', '5000',
    '--num-epochs', '1',
]

print('+', ' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)

## Full Quick Run

Fixed dVAE + seqrec for both encoders, close to `01_run_experiments.ipynb`: dVAE 5 epochs, seqrec `history_budget=128`, `depth=4`, `beam_size=50`.

In [ ]:
full_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.run_encoder_experiment',
    '--interactions', str(INTERACTIONS),
    '--embedding', f'e5={EMBEDDINGS["e5"]}',
    '--embedding', f'bge={EMBEDDINGS["bge"]}',
    '--test-interval-days', '7',
    '--work-data-dir', 'data/RQ_encoder_comparison/yambda_fixed',
    '--results-dir', 'results/RQ_encoder_comparison_yambda_fixed',
    '--dvae-config', 'configs/RQ_encoder_comparison/yambda_fixed_dvae.yaml',
    '--seqrec-config', 'configs/RQ_encoder_comparison/seqrec_yambda_fixed.yaml',
    '--stages', 'prepare,dvae,sid_metrics,seqrec',
    '--max-users', '5000',
    '--max-core-items', '30000',
]

print('+', ' '.join(full_cmd))
subprocess.run(full_cmd, check=True)

## Collect Results

In [ ]:
collect_cmd = [
    PY, '-m', 'scripts.RQ_encoder_comparison.collect_results',
    '--results-dir', 'results/RQ_encoder_comparison_yambda_fixed',
]

print('+', ' '.join(collect_cmd))
subprocess.run(collect_cmd, check=True)

In [ ]:
import pandas as pd

pd.read_csv(ROOT / 'results' / 'RQ_encoder_comparison_yambda_fixed' / 'comparison.csv')